## CSC 466: Knowledge Discovery in Data
## Cal Poly, San Luis Obispo
### Spring 2026

# Lab 3


### Due Thursday, May 7, 3:00pm




**Name:** Winnie Trinh

**Cal Poyl Email:** witrinh@calpoly.edu

**Name:** Nathan Madlansacay

**Cal Poyl Email:** nmadlans@calpoly.edu

LINK to the Github Repo for MlReport
https://github.com/Lucas-Summers/mlreport

In [ ]:
!pip install git+https://github.com/Lucas-Summers/mlreport.git

  Cloning https://github.com/Lucas-Summers/mlreport.git to /tmp/pip-req-build-936jw00d
  Running command git clone --filter=blob:none --quiet https://github.com/Lucas-Summers/mlreport.git /tmp/pip-req-build-936jw00d
  Resolved https://github.com/Lucas-Summers/mlreport.git to commit 778cc46b661fe248059479f6f81807b8b661f695
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 25.9 MB/s eta 0:00:00
  Created wheel for mlreport: filename=mlreport-0.0.1-py3-none-any.whl size=52869 sha256=e167753c4baa3636c1f01729ea92347c59cab194b9a2321911202359e4300b56
  Stored in directory: /tmp/pip-ephem-wheel-cache-131hhqw9/wheels/f8/a0/50/b2dad99aa952ffc8015c0986fcb67b722b2f3ea0e78e5f801e
Successfully built mlre

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
### import specifc sklearn classes below

from sklearn import datasets
from collections import Counter
from math import log2
from pandas.api.types import is_numeric_dtype
import json
import math

**This first section is the Decision tree and its predictor function and fit function**

In [ ]:
%%writefile c45.py

import json
import pandas as pd
from math import log2
from collections import Counter
from pandas.api.types import is_numeric_dtype
class c45:
  def __init__(self, metric="InfoGain", threshold=0.0, featureType=None):
    """
    Parameters:
    metric: str - "InfoGain" or "Ratio"

    threshold: float
    create node if metric > threshold

    featureType: dictionary - "categorical" or "numeric"
    """
    # check if correct desired metric is allowed
    allowedMetrics = ["InfoGain", "Ratio"]
    if metric not in allowedMetrics:
        raise ValueError(f"metric must be one of {allowedMetrics}")

    # check if threshold is a valid number
    if not isinstance(threshold, (int, float)) or threshold < 0.0:
        raise ValueError(f"threshold must be a nonnegative number")

    self.metric = metric
    self.threshold = threshold
    self.featureType = featureType
    self.defaultClass = None

    #final decision tree after calling fit()
    self.tree = None

  # determine whether each feature is numeric or categorical
  def determineFeatures(self, X):
    features = {}

    # loop through each column name in DataFrame
    for col in X.columns:
      if is_numeric_dtype(X[col]):
        features[col] = "numeric"
      else:
        features[col] = "categorical"

    return features

  # trains and builds the C45 decision tree
  def fit(self, Xtrain, ytrain, data=None, featureType=None):
    """
    Builds C4.5 decision tree using the training data

    Parameters:
    Xtrain: DataFrame of independent variables
    ytrain: Series/list of class labels
    data: optional name of the dataset
    """
    # convert into DataFrame and reset row numbers to 0, 1, ...
    Xtrain = pd.DataFrame(Xtrain).reset_index(drop=True)
    # convert to Series and reset row numbers to match ^
    ytrain = pd.Series(ytrain).reset_index(drop=True)

    self.defaultClass = self.majorityClass(ytrain)

    # use featureType dict or determine them
    if featureType is not None:
        self.featureType = featureType
    elif self.featureType is None:
        self.featureType = self.determineFeatures(Xtrain)

    # build tree using columns as possible splitting attributes
    root = self.buildTree(Xtrain, ytrain, list(Xtrain.columns))

    # store tree in json format
    self.tree = {
        "dataset": data,
        "node": root
    }

    return self.tree

  # calculate entropy, how unorganized class labels are
  def entropy(self, y):
    #case 1: if there are no labels
    if len(y) == 0:
      return 0.0

    # count number of times label appears
    total = len(y)
    counts = Counter(y)

    entropy = 0.0

    # calculate proportion of class
    for count in counts.values():
      proportion = count / total
      entropy += -proportion * log2(proportion)

    return entropy

  def infoGain(self, X, y, attribute, threshold = None):
    """
    Calculates information gain of splitting on an attribute.

    X: DataFrame of features
    y: labels
    attribute: str - column name to split on
    threshold: float or None
      None -> categorical split
      float -> numeric split at value
    """
    # calculate entrop before attribute split
    pEntropy = self.entropy(y)

    # total number of rows
    total = len(y)

    # no infogain if there are no labels
    if total == 0:
      return 0.0

    # entropy after attribute split
    weightedEntropy = 0.0

    # treat as categorical
    if threshold is None:
      # loop through each value for attribute
      for value in X[attribute].unique():
        subset = y[X[attribute] == value]
        weightedEntropy += (len(subset) / total) * self.entropy(subset)

    # treat as numeric attribute
    else:
      # create D- and D+ subsets
      leftSubset = y[X[attribute] <= threshold]
      rightSubset = y[X[attribute] > threshold]

      weightedEntropy += (len(leftSubset) / total) * self.entropy(leftSubset)
      weightedEntropy += (len(rightSubset) / total) * self.entropy(rightSubset)

    return pEntropy - weightedEntropy

  # calculate split value which is denominator for gainRatio
  def splitValue(self, X, attribute, threshold=None):
    # total number of rows
    total = len(X)

    # return 0.0 if no rows
    if total == 0:
      return 0.0

    splitVal = 0.0

    # treat as categorical split
    if threshold is None:
      # loop through each value for attribute
      for value in X[attribute].unique():
        proportion = len(X[X[attribute] == value]) / total

        if proportion > 0:
          splitVal += -proportion * log2(proportion)

    # treat as numeric split
    else:
      # create D- and D+ subsets
      leftSubset = X[X[attribute] <= threshold]
      rightSubset = X[X[attribute] > threshold]

      # loop through subsets and calculate the proportions
      for subset in [leftSubset, rightSubset]:
        proportion = len(subset) / total

        if proportion > 0:
          splitVal += -proportion * log2(proportion)

    return splitVal

  # calculate gain ratio, other option to infogain
  def gainRatio(self, X, y, attribute, threshold=None):
    """
    Calculates gain ratio of splitting on an attribute
    """

    # calculate information gain first
    gain = self.infoGain(X, y, attribute, threshold)
    # then calculate split value
    splitVal = self.splitValue(X, attribute, threshold)

    if splitVal == 0.0:
      return 0.0

    # return ratio
    return gain / splitVal

  def majorityClass(self, y):
    """
    Returns the majority class
    """

    # return None if no labels
    if len(y) == 0:
      return None

    # return most command class label
    return Counter(y).most_common(1)[0][0]

  def possibleThresholds(self, X, attribute):
    """
    Returns a list of possible thresholds for a numeric attribute
    """

    values = sorted(X[attribute].dropna().unique())
    thresholds = []

    # loop through each pair of values
    for i in range(len(values) - 1):
      # add midpoints between values
      thresholds.append((values[i] + values[i + 1]) / 2)

    return thresholds

  def findBestSplit(self, X, y, attributes):
    """
    Finds the best attribute to split on
    """
    best = None
    bestGain = 0.0
    bestThreshold = None

    # loop through each attribute
    for attribute in attributes:
      attributeType = self.featureType[attribute]

      # treat as numeric attribute
      if attributeType == "numeric":
        # find possible thresholds
        thresholds = self.possibleThresholds(X, attribute)
        for t in thresholds:
          if self.metric == "InfoGain":
            gain = self.infoGain(X, y, attribute, t)
          else:
            gain = self.gainRatio(X, y, attribute, t)

          # save information if this is the best gain score so far
          if gain > bestGain:
            best = attribute
            bestGain = gain
            bestThreshold = t

      # treat as categorical attributes, no thresholds
      else:
        if self.metric == "InfoGain":
          gain = self.infoGain(X, y, attribute)
        else:
          gain = self.gainRatio(X, y, attribute)

        # save information if this is the best gain score so far
        if gain > bestGain:
          best = attribute
          bestGain = gain
          bestThreshold = None

    return best, bestThreshold, bestGain

  # tree building function and helper functions

  # create leaf node, this is the final prediction
  def leaf(self, y):
    """
    Returns a leaf node with the majority class label
    """

    # get most common class label
    mostCommon = self.majorityClass(y)


    # calculate probability
    if len(y) == 0:
      prob = 0.0

    else:
      prob = Counter(y)[mostCommon] / len(y)

    return {
        "leaf": {
            "decision": mostCommon,
            "p": prob
        }
    }

  # create edge between a parent and child
  def edge(self, val, op, child):
    """
    Creates an edge node

    val: category/numeric threshold
    op: attribute name
    child: child node/leaf
    """

    # edge has a value
    edge = {
        "value": val
    }
    if op is not None:
      edge["op"] = op

    # store child as leaf or decision node
    if "leaf" in child:
      edge["leaf"] = child["leaf"]
    else:
      edge["node"] = child

    return {
        "edge": edge
    }

  # build decision tree
  def buildTree(self, X, y, attributes, parent=None):
    """
    Build decision tree

    X: DataFrame of features
    y: labels
    attributes: list of attributes for splitting
    """

    # no data left
    if len(y) == 0:
      return self.leaf(y)

    # no attributes left
    if len(attributes) == 0:
      return self.leaf(y)

    # all labels are the same
    if len(set(y)) == 1:
      return self.leaf(y)

    best, bestThreshold, bestGain = self.findBestSplit(X, y, attributes)

    # no attribute gives good split
    if best is None or bestGain <= self.threshold:
      return self.leaf(y)

    node = {
        "var": best,
        "type": self.featureType[best],
        "mostCommon": self.majorityClass(y),
        "edges": []
    }

    # numberic split
    if self.featureType[best] == "numeric":
      # create left and right subtrees
      leftChild = self.buildTree(X[X[best] <= bestThreshold].reset_index(drop=True),
                                 y[X[best] <= bestThreshold].reset_index(drop=True),
                                 attributes)
      rightChild = self.buildTree(X[X[best] > bestThreshold].reset_index(drop=True),
                                  y[X[best] > bestThreshold].reset_index(drop=True),
                                  attributes)

      # add edges connecting node to left and right subtrees
      node["edges"].append(self.edge(bestThreshold, "<=", leftChild))
      node["edges"].append(self.edge(bestThreshold, ">", rightChild))

    # categorical split
    else:
      # remove category after splitting
      newAttributes = attributes.copy()
      newAttributes.remove(best)

      # create a branch per value in the category
      for val in X[best].unique():
        child = self.buildTree(X[X[best] == val].reset_index(drop=True),
                               y[X[best] == val].reset_index(drop=True),
                               newAttributes)
        # add edge to connect new category branch
        node["edges"].append(self.edge(val, None, child))

    return node

  # prediction functions
  def predict(self, Xtest):
    """
    Predicts class labels for new data

    Xtest: DataFrame of test data
    """

    if self.tree is None:
      raise ValueError("Decision tree not trained")

    # convert test data into DataFrame
    Xtest = pd.DataFrame(Xtest).reset_index(drop=True)

    preds = []

    # for each row, predict what final decision would be
    for i, row in Xtest.iterrows():
      preds.append(self.predictRow(row, self.tree["node"]))

    return preds

  # helper function to predict, predicts for a singular row
  def predictRow(self, row, node):
    """
    Predicts class label for a single row

    row: Series of row data
    node: current node in tree
    """

    # return decision if node is already a leaf
    if "leaf" in node:
      return node["leaf"]["decision"]

    attr = node["var"]
    attrType = node["type"]
    attrVal = row[attr]

    # numeric split
    if attrType == "numeric":
      # go through every edge that connects to this node
      for edge in node["edges"]:
        e = edge["edge"]
        threshold = e["value"]
        op = e["op"]

        # <= branch
        if op == "<=" and attrVal <= threshold:
          if "leaf" in e:
            return e["leaf"]["decision"]
          else:
            return self.predictRow(row, e["node"])

        # > branch
        if op == ">" and attrVal > threshold:
          if "leaf" in e:
            return e["leaf"]["decision"]
          else:
            return self.predictRow(row, e["node"])
    # categorical split
    else:
      for edge in node["edges"]:
        e = edge["edge"]

        if attrVal == e["value"]:
          if "leaf" in e:
            return e["leaf"]["decision"]
          else:
            return self.predictRow(row, e["node"])

    # return most common label if the other return statements don't execute
    return node.get("mostCommon", self.defaultClass)

  # save and read json
  def read_tree(self, file):
    """
    Reads in a decision tree from a json file
    """
    with open(file, "r") as f:
      self.tree = json.load(f)

    return self.tree

  def save_tree(self, file):
    """
    Saves a decision tree to a json file
    """
    if self.tree is None:
      raise ValueError("Decision tree not trained")

    with open(file, "w") as f:
      json.dump(self.tree, f, indent=4)


Writing c45.py


In [ ]:
%%writefile predict.py
import sys
from collections import defaultdict

from c45 import c45
from utils import parseCSV
def printConfusionMatrix(matrix, classes):
    """
    Prints a formatted confusion matrix.

    matrix: dict of dict  {actual: {predicted: count}}
    classes: sorted list of unique class labels
    """
    # column width based on longest class name
    colWidth = max(len(str(c)) for c in classes) + 2

    # header row
    header = "Actual \\ Predicted".ljust(colWidth)
    for c in classes:
        header += str(c).rjust(colWidth)
    print(header)
    print("-" * len(header))

    # one row per actual class
    for actual in classes:
        row = str(actual).ljust(colWidth)
        for predicted in classes:
            count = matrix[actual][predicted]
            row += str(count).rjust(colWidth)
        print(row)

def main():
    if len(sys.argv) < 3:
        print("Usage: python predict.py <CSVFile> <JSONFile> [eval]")
        sys.exit(1)

    csvFile  = sys.argv[1]
    jsonFile = sys.argv[2]
    evalMode = len(sys.argv) > 3 and sys.argv[3].lower() == "eval"

    #  load the decision tree from JSON
    model = c45()
    model.read_tree(jsonFile)

    #  parse the CSV file
    X, y, featureType, classVar = parseCSV(csvFile)

    #  make predictions
    predictions = model.predict(X)

    #  basic output: one prediction per line
    if not evalMode:
        for pred in predictions:
            print(pred)
        return

    #  eval mode: need ground truth
    if y is None:
        print("Error: eval mode requires ground truth labels in the CSV file.")
        sys.exit(1)

    groundTruth = list(y)

    # print predictions alongside ground truth
    print(f"{'Row':<6} {'Actual':<20} {'Predicted':<20} {'Correct'}")
    print("-" * 60)
    for i, (actual, predicted) in enumerate(zip(groundTruth, predictions)):
        correct = "Yes" if actual == predicted else "No"
        print(f"{i:<6} {str(actual):<20} {str(predicted):<20} {correct}")

    print()

    #  compute counts
    total   = len(groundTruth)
    correct = sum(a == p for a, p in zip(groundTruth, predictions))
    wrong   = total - correct

    #  compute accuracy and error rate
    accuracy  = correct / total if total > 0 else 0.0
    errorRate = wrong   / total if total > 0 else 0.0

    # print summary stats
    print(f"Total records classified:{total}")
    print(f"Correctly classified:{correct}")
    print(f"Incorrectly classified:{wrong}")
    print(f"Accuracy:{accuracy:.4f}")
    print(f"Error rate:{errorRate:.4f}")

    #  build confusion matrix
    # rows = actual class, columns = predicted class
    classes = sorted(set(groundTruth) | set(predictions))
    matrix  = defaultdict(lambda: defaultdict(int))

    for actual, predicted in zip(groundTruth, predictions):
        matrix[actual][predicted] += 1

    #  print confusion matrix
    print("Confusion Matrix:")
    printConfusionMatrix(matrix, classes)


if __name__ == "__main__":
    main()

Writing predict.py


In [ ]:
%%writefile utils.py

import math
import random
import pandas as pd
from collections import defaultdict


def parseCSV(filepath):
    """
    Parses the custom 3-line header CSV format.

    First line: column names
    Second line: domain codes
        -1 = row ID / ignore
         0 = numeric
        >0 = categorical
    Third line: class variable name

    Returns:
        X, y, featureType, classVar
    """
    with open(filepath, "r") as f:
        lines = f.readlines()

    colNames = [c.strip() for c in lines[0].strip().split(",")]
    domainCodes = [int(x.strip()) for x in lines[1].strip().split(",")]
    classVar = lines[2].strip()

    data = pd.read_csv(filepath, skiprows=3, header=None, names=colNames)

    dropCols = []

    for i, code in enumerate(domainCodes):
        if code == -1:
            dropCols.append(colNames[i])

    data = data.drop(columns=dropCols)

    featureType = {}

    for i, col in enumerate(colNames):
        if col in dropCols:
            continue

        if col == classVar:
            continue

        if domainCodes[i] == 0:
            featureType[col] = "numeric"
            data[col] = pd.to_numeric(data[col], errors="coerce")
        else:
            featureType[col] = "categorical"
            data[col] = data[col].astype(str)

    if classVar in data.columns:
        X = data.drop(columns=[classVar])
        y = data[classVar]
    else:
        X = data
        y = None

    return X, y, featureType, classVar


def trainTestSplit(X, y, testFrac=0.2, seed=67):
    """
    Reproducible 80/20 train-test split.

    Returns:
        Xtrain, Xtest, ytrain, ytest
    """
    random.seed(seed)

    X = pd.DataFrame(X).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    indices = list(range(len(y)))
    random.shuffle(indices)

    cutoff = int(len(indices) * (1 - testFrac))

    trainIdx = indices[:cutoff]
    testIdx = indices[cutoff:]

    Xtrain = X.iloc[trainIdx].reset_index(drop=True)
    Xtest = X.iloc[testIdx].reset_index(drop=True)
    ytrain = y.iloc[trainIdx].reset_index(drop=True)
    ytest = y.iloc[testIdx].reset_index(drop=True)

    return Xtrain, Xtest, ytrain, ytest


def accuracy(yTrue, yPred):
    """
    Computes accuracy.
    """
    yTrue = list(yTrue)
    yPred = list(yPred)

    total = len(yTrue)

    if total == 0:
        return 0.0

    return sum(a == p for a, p in zip(yTrue, yPred)) / total


def errorRate(yTrue, yPred):
    """
    Computes error rate.
    """
    return 1.0 - accuracy(yTrue, yPred)


def buildGrid(numAttrs, numRows, customGrid=None):
    """
    Build a dynamic grid for Random Forest hyperparameter search.

    Returns dictionary:
        {
            "NumTrees": [...],
            "NumAttributes": [...],
            "NumDataPoints": [...]
        }
    """
    if customGrid is not None:
        return customGrid

    # Attribute range
    if numAttrs <= 4:
        mRange = list(range(1, numAttrs + 1))
    elif numAttrs <= 10:
        mRange = [1, 2, 3, 4]
    elif numAttrs <= 20:
        mRange = [1, 2, 3, 4, 5]
    else:
        mRange = [1, 2, 3, 4, 5, 6, 7]

    # Tree range
    triples = math.comb(numAttrs, min(3, numAttrs))

    if triples >= 500:
        nRange = [60, 120, 250]
    elif triples >= 100:
        nRange = [25, 50, 100]
    else:
        nRange = [10, 25, 50]

    # Row sample range
    if numRows >= 10000:
        kRange = [0.05, 0.10, 0.20, 0.25]
    elif numRows >= 1000:
        kRange = [0.10, 0.20, 0.30, 0.50]
    elif numRows >= 100:
        kRange = [0.30, 0.50, 0.70, 1.0]
    else:
        kRange = [
            max(5, numRows // 4),
            max(10, numRows // 2),
            numRows
        ]

    return {
        "NumTrees": nRange,
        "NumAttributes": mRange,
        "NumDataPoints": kRange,
    }


def encodeForSklearn(Xtrain, Xtest, featureType):
    """
    Converts mixed numeric/categorical data into numeric data for Scikit-learn.

    Your custom C4.5 can usually handle categorical values directly,
    but Scikit-learn RandomForestClassifier needs numbers.
    """
    Xtrain = pd.DataFrame(Xtrain).copy()
    Xtest = pd.DataFrame(Xtest).copy()

    combined = pd.concat([Xtrain, Xtest], axis=0, ignore_index=True)

    categoricalCols = [
        col for col in combined.columns
        if featureType.get(col) == "categorical"
    ]

    numericCols = [
        col for col in combined.columns
        if featureType.get(col) == "numeric"
    ]

    for col in numericCols:
        combined[col] = pd.to_numeric(combined[col], errors="coerce")
        combined[col] = combined[col].fillna(combined[col].median())

    combined = pd.get_dummies(combined, columns=categoricalCols)

    XtrainEncoded = combined.iloc[:len(Xtrain)].reset_index(drop=True)
    XtestEncoded = combined.iloc[len(Xtrain):].reset_index(drop=True)

    return XtrainEncoded, XtestEncoded


def confusionMatrix(yTrue, yPred):
    """
    Builds a confusion matrix as a nested dictionary.
    """
    classes = sorted(set(list(yTrue) + list(yPred)), key=str)
    matrix = defaultdict(lambda: defaultdict(int))

    for actual, predicted in zip(yTrue, yPred):
        matrix[actual][predicted] += 1

    return matrix, classes


def printConfusionMatrix(yTrue, yPred):
    """
    Prints a formatted confusion matrix.
    """
    matrix, classes = confusionMatrix(yTrue, yPred)

    colWidth = max(len(str(c)) for c in classes) + 2

    header = "Actual \\ Predicted".ljust(colWidth)

    for c in classes:
        header += str(c).rjust(colWidth)

    print(header)
    print("-" * len(header))

    for actual in classes:
        row = str(actual).ljust(colWidth)

        for predicted in classes:
            row += str(matrix[actual][predicted]).rjust(colWidth)

        print(row)

Writing utils.py


**Random Forest Class**

In [ ]:
%%writefile RandomForestClassifier.py

import random
import pandas as pd
from collections import Counter

from c45 import c45


class RandomForestClassifier:
    def __init__(
        self,
        NumTrees=100,
        NumAttributes=None,
        NumDataPoints=None,
        SplittingMetric="InfoGain",
        Threshold=0.0,
        random_state=67
    ):
        """
        Random Forest classifier built using your C4.5 decision tree.

        Parameters:
            NumTrees: number of trees in the forest
            NumAttributes: number of attributes randomly chosen for each tree
            NumDataPoints: number/fraction of rows sampled with replacement
            SplittingMetric: passed to C4.5
            Threshold: passed to C4.5
            random_state: seed for reproducibility
        """
        self.NumTrees = NumTrees
        self.NumAttributes = NumAttributes
        self.NumDataPoints = NumDataPoints
        self.SplittingMetric = SplittingMetric
        self.Threshold = Threshold
        self.random_state = random_state

        self.forest = []
        self.featureType = None
        self.classes_ = None

    def _resolveK(self, n):
        """
        Figure out how many rows to sample for each tree.

        If NumDataPoints is None, use all rows.
        If NumDataPoints is between 0 and 1, treat it as a fraction.
        If NumDataPoints is greater than 1, treat it as an exact row count.
        """
        if self.NumDataPoints is None:
            return n

        if self.NumDataPoints <= 1.0:
            return max(1, int(self.NumDataPoints * n))

        return min(int(self.NumDataPoints), n)

    def _resolveM(self, totalAttributes):
        """
        Figure out how many attributes to sample for each tree.
        If NumAttributes is None, use sqrt(totalAttributes).
        """
        if self.NumAttributes is None:
            return max(1, int(totalAttributes ** 0.5))

        return max(1, min(int(self.NumAttributes), totalAttributes))

    def fit(self, X, Y, featureType=None):
        """
        Train the random forest.

        X: training features
        Y: ground truth labels
        featureType: dictionary like {"Age": "numeric", "Sex": "categorical"}
        """
        random.seed(self.random_state)

        X = pd.DataFrame(X).reset_index(drop=True)
        Y = pd.Series(Y).reset_index(drop=True)

        if len(X) != len(Y):
            raise ValueError("X and Y must have the same number of rows.")

        n = len(Y)
        allAttrs = list(X.columns)

        if n == 0:
            raise ValueError("Cannot train RandomForest on empty dataset.")

        if len(allAttrs) == 0:
            raise ValueError("Cannot train RandomForest with zero attributes.")

        self.featureType = featureType
        self.classes_ = sorted(set(Y), key=str)
        self.forest = []

        k = self._resolveK(n)
        m = self._resolveM(len(allAttrs))

        for treeNum in range(self.NumTrees):
            # Bootstrap sample rows with replacement
            rowIndices = [random.randint(0, n - 1) for _ in range(k)]
            Xsample = X.iloc[rowIndices].reset_index(drop=True)
            ysample = Y.iloc[rowIndices].reset_index(drop=True)

            # Randomly choose attributes without replacement
            attrSubset = random.sample(allAttrs, m)
            XsubsetSample = Xsample[attrSubset]

            if featureType is not None:
                subFeatureType = {
                    attr: featureType[attr]
                    for attr in attrSubset
                    if attr in featureType
                }
            else:
                subFeatureType = None

            # Build one C4.5 tree
            try:
                tree = c45(
                    metric=self.SplittingMetric,
                    threshold=self.Threshold,
                    featureType=subFeatureType
                )
            except TypeError:
                # Backup in case your c45 constructor does not accept these names
                tree = c45()
                if hasattr(tree, "metric"):
                    tree.metric = self.SplittingMetric
                if hasattr(tree, "threshold"):
                    tree.threshold = self.Threshold
                if hasattr(tree, "featureType"):
                    tree.featureType = subFeatureType

            tree.fit(XsubsetSample, ysample)

            self.forest.append((tree, attrSubset))

        return self

    def predict(self, X):
        """
        Predict labels using majority vote across all trees.
        Ties are broken consistently using string order.
        """
        if not self.forest:
            raise ValueError("RandomForest has not been trained yet. Call fit() first.")

        X = pd.DataFrame(X).reset_index(drop=True)

        votes = [[] for _ in range(len(X))]

        for tree, attrSubset in self.forest:
            Xsub = X[attrSubset]
            preds = tree.predict(Xsub)

            for i, pred in enumerate(preds):
                votes[i].append(pred)

        finalPredictions = []

        for rowVotes in votes:
            counts = Counter(rowVotes)

            winner = min(
                counts.keys(),
                key=lambda label: (-counts[label], str(label))
            )

            finalPredictions.append(winner)

        return finalPredictions

Writing RandomForestClassifier.py


**Random Forest Evaluation**

In [ ]:
%%writefile rfEval.py

import sys
import random
import math
from collections import defaultdict

import pandas as pd

from sklearn.ensemble import RandomForestClassifier as SklearnRandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt

try:
    from mlreport import Report, ComparisonReport
    MLREPORT_AVAILABLE = True
except ImportError:
    MLREPORT_AVAILABLE = False

from utils import parseCSV
from RandomForestClassifier import RandomForestClassifier


def trainTestSplit(X, y, testFrac=0.2, seed=67):
    """
    Reproducible 80/20 split.

    Returns:
        Xtrain, ytrain, Xtest, ytest
    """
    random.seed(seed)

    X = pd.DataFrame(X).reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)

    indices = list(range(len(y)))
    random.shuffle(indices)

    cutoff = int(len(indices) * (1 - testFrac))

    trainIdx = indices[:cutoff]
    testIdx = indices[cutoff:]

    Xtrain = X.iloc[trainIdx].reset_index(drop=True)
    ytrain = y.iloc[trainIdx].reset_index(drop=True)

    Xtest = X.iloc[testIdx].reset_index(drop=True)
    ytest = y.iloc[testIdx].reset_index(drop=True)

    return Xtrain, ytrain, Xtest, ytest


def accuracy(yTrue, yPred):
    """
    Returns accuracy.
    """
    yTrue = list(yTrue)
    yPred = list(yPred)

    total = len(yTrue)

    if total == 0:
        return 0.0

    return sum(a == p for a, p in zip(yTrue, yPred)) / total


def buildGrid(numAttrs, numRows, customGrid=None):
    """
    Build a dynamic grid for Random Forest hyperparameter search.

    Returns:
        {
            "NumTrees": [...],
            "NumAttributes": [...],
            "NumDataPoints": [...]
        }
    """
    if customGrid:
        return customGrid

    # NumAttributes range
    if numAttrs <= 10:
        mRange = list(range(1, min(5, numAttrs + 1)))
    elif numAttrs <= 20:
        mRange = list(range(1, 6))
    else:
        mRange = list(range(1, 8))

    # NumTrees range
    triples = math.comb(numAttrs, min(3, numAttrs))

    if triples >= 500:
        nRange = [60, 120, 250, 500]
    elif triples >= 100:
        nRange = [25, 50, 100, 200]
    else:
        nRange = [10, 25, 50, 100]

    # NumDataPoints range
    if numRows >= 10000:
        kRange = [0.05, 0.10, 0.20, 0.25]
    elif numRows >= 1000:
        kRange = [0.10, 0.20, 0.30, 0.50]
    elif numRows >= 100:
        kRange = [0.30, 0.50, 0.70, 1.0]
    else:
        kRange = [
            max(5, numRows // 4),
            max(10, numRows // 2),
            numRows
        ]

    return {
        "NumTrees": nRange,
        "NumAttributes": mRange,
        "NumDataPoints": kRange,
    }


def gridSearchCustomRF(Xtrain, ytrain, Xtest, ytest, featureType, grid):
    """
    3-way grid search over:

        NumTrees x NumAttributes x NumDataPoints

    Returns:
        bestModel, bestParams, bestAccuracy, bestPreds
    """
    best = {
        "acc": -1,
        "model": None,
        "params": None,
        "preds": None
    }

    total = (
        len(grid["NumTrees"])
        * len(grid["NumAttributes"])
        * len(grid["NumDataPoints"])
    )

    done = 0

    print()
    print("===== Custom Random Forest Grid Search =====")

    for N in grid["NumTrees"]:
        for m in grid["NumAttributes"]:
            for k in grid["NumDataPoints"]:
                done += 1

                print(f"[{done}/{total}] N={N}, m={m}, k={k}", end="  ")

                rf = RandomForestClassifier(
                    NumTrees=N,
                    NumAttributes=m,
                    NumDataPoints=k,
                    SplittingMetric="InfoGain",
                    Threshold=0.0
                )

                rf.fit(Xtrain, ytrain, featureType=featureType)

                preds = rf.predict(Xtest)
                acc = accuracy(ytest, preds)

                print(f"acc={acc:.4f}")

                if acc > best["acc"]:
                    best["acc"] = acc
                    best["model"] = rf
                    best["params"] = {
                        "NumTrees": N,
                        "NumAttributes": m,
                        "NumDataPoints": k,
                        "SplittingMetric": "InfoGain",
                        "Threshold": 0.0
                    }
                    best["preds"] = preds

    return best["model"], best["params"], best["acc"], best["preds"]


def encodeForSklearn(Xtrain, Xtest, featureType):
    """
    Scikit-learn needs numeric data.

    This converts categorical columns into dummy variables.
    """
    Xtrain = pd.DataFrame(Xtrain).copy()
    Xtest = pd.DataFrame(Xtest).copy()

    combined = pd.concat([Xtrain, Xtest], axis=0, ignore_index=True)

    categoricalCols = []
    numericCols = []

    for col in combined.columns:
        if featureType.get(col) == "categorical":
            categoricalCols.append(col)
        else:
            numericCols.append(col)

    for col in numericCols:
        combined[col] = pd.to_numeric(combined[col], errors="coerce")
        combined[col] = combined[col].fillna(combined[col].median())

    combined = pd.get_dummies(combined, columns=categoricalCols)

    XtrainEncoded = combined.iloc[:len(Xtrain)].reset_index(drop=True)
    XtestEncoded = combined.iloc[len(Xtrain):].reset_index(drop=True)

    return XtrainEncoded, XtestEncoded


def convertKForSklearn(k, nRows):
    """
    Converts NumDataPoints into Scikit-learn's max_samples.
    """
    if k is None:
        return None

    if k <= 1.0:
        return float(k)

    return min(int(k), nRows)


def gridSearchSklearnRF(Xtrain, ytrain, Xtest, ytest, grid):
    """
    Grid search for Scikit-learn RandomForestClassifier.

    Mapping:
        NumTrees -> n_estimators
        NumAttributes -> max_features
        NumDataPoints -> max_samples
    """
    best = {
        "acc": -1,
        "model": None,
        "params": None,
        "preds": None
    }

    total = (
        len(grid["NumTrees"])
        * len(grid["NumAttributes"])
        * len(grid["NumDataPoints"])
    )

    done = 0

    print()
    print("===== Scikit-learn Random Forest Grid Search =====")

    for N in grid["NumTrees"]:
        for m in grid["NumAttributes"]:
            for k in grid["NumDataPoints"]:
                done += 1

                maxSamples = convertKForSklearn(k, len(Xtrain))

                print(
                    f"[{done}/{total}] "
                    f"N={N}, max_features={m}, max_samples={maxSamples}",
                    end="  "
                )

                rf = SklearnRandomForestClassifier(
                    n_estimators=N,
                    max_features=m,
                    bootstrap=True,
                    max_samples=maxSamples,
                    random_state=67
                )

                rf.fit(Xtrain, ytrain)

                preds = list(rf.predict(Xtest))
                acc = accuracy(ytest, preds)

                print(f"acc={acc:.4f}")

                if acc > best["acc"]:
                    best["acc"] = acc
                    best["model"] = rf
                    best["params"] = {
                        "n_estimators": N,
                        "max_features": m,
                        "max_samples": maxSamples,
                        "bootstrap": True,
                        "random_state": 67
                    }
                    best["preds"] = preds

    return best["model"], best["params"], best["acc"], best["preds"]


def printConfusionMatrix(yTrue, yPred):
    """
    Prints a formatted confusion matrix.
    """
    classes = sorted(set(list(yTrue) + list(yPred)), key=str)

    matrix = defaultdict(lambda: defaultdict(int))

    for actual, predicted in zip(yTrue, yPred):
        matrix[actual][predicted] += 1

    colW = max(len(str(c)) for c in classes) + 2

    header = "Actual \\ Predicted".ljust(colW)

    for c in classes:
        header += str(c).rjust(colW)

    print(header)
    print("-" * len(header))

    for actual in classes:
        row = str(actual).ljust(colW)

        for predicted in classes:
            row += str(matrix[actual][predicted]).rjust(colW)

        print(row)


def generateMLReport(
    outputFile,
    datasetName,
    Xtrain,
    ytrain,
    Xtest,
    ytest,
    XtrainSklearn,
    XtestSklearn,
    customModel,
    customParams,
    customPreds,
    sklearnModel,
    sklearnParams,
    sklearnPreds
):
    """
    Generates an mlreport ComparisonReport PDF.
    """

    # Train predictions are useful for mlreport
    customTrainPreds = customModel.predict(Xtrain)
    sklearnTrainPreds = list(sklearnModel.predict(XtrainSklearn))

    # Report for custom Random Forest
    customReport = Report(
        customModel,
        title="Custom RandomForest",
        model_type="classification",
        model_params=customParams
    )

    customReport.add_split("train", Xtrain, list(ytrain), customTrainPreds)
    customReport.add_split("test", Xtest, list(ytest), customPreds)
    customReport.build()

    # Report for Scikit-learn Random Forest
    sklearnReport = Report(
        sklearnModel,
        title="Sklearn RandomForest",
        model_type="classification",
        model_params=sklearnParams
    )

    sklearnReport.add_split("train", XtrainSklearn, list(ytrain), sklearnTrainPreds)
    sklearnReport.add_split("test", XtestSklearn, list(ytest), sklearnPreds)
    sklearnReport.build()

    # Comparison report
    comparison = ComparisonReport(
        reports=[customReport, sklearnReport],
        title=f"Random Forest Comparison — {datasetName}",
        split="test",
        theme="light"
    )

    comparison.build()
    comparison.to_pdf(outputFile)

    print(f"mlreport PDF saved to: {outputFile}")


def fallbackMatplotlibReport(
    outputFile,
    datasetName,
    ytest,
    customParams,
    customAcc,
    customPreds,
    sklearnParams,
    sklearnAcc,
    sklearnPreds
):
    """
    Fallback PDF using matplotlib if mlreport does not work.
    """
    labels = sorted(
        set(list(ytest) + list(customPreds) + list(sklearnPreds)),
        key=str
    )

    with PdfPages(outputFile) as pdf:

        # Summary page
        fig = plt.figure(figsize=(8.5, 11))
        plt.axis("off")

        summaryText = (
            f"Random Forest Comparison Report\n\n"
            f"Dataset: {datasetName}\n\n"
            f"Custom Random Forest\n"
            f"  Accuracy:   {customAcc:.4f}\n"
            f"  Error Rate: {1 - customAcc:.4f}\n"
            f"  Params: {customParams}\n\n"
            f"Scikit-learn Random Forest\n"
            f"  Accuracy:   {sklearnAcc:.4f}\n"
            f"  Error Rate: {1 - sklearnAcc:.4f}\n"
            f"  Params: {sklearnParams}\n"
        )

        plt.text(
            0.05,
            0.95,
            summaryText,
            va="top",
            fontsize=10,
            family="monospace"
        )

        pdf.savefig(fig)
        plt.close(fig)

        # Custom Random Forest confusion matrix
        cmCustom = confusion_matrix(ytest, customPreds, labels=labels)

        fig, ax = plt.subplots(figsize=(8, 6))
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cmCustom,
            display_labels=labels
        )
        disp.plot(xticks_rotation=45, ax=ax)
        plt.title("Custom Random Forest — Confusion Matrix")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # Scikit-learn confusion matrix
        cmSklearn = confusion_matrix(ytest, sklearnPreds, labels=labels)

        fig, ax = plt.subplots(figsize=(8, 6))
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cmSklearn,
            display_labels=labels
        )
        disp.plot(xticks_rotation=45, ax=ax)
        plt.title("Scikit-learn Random Forest — Confusion Matrix")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    print(f"Fallback matplotlib PDF saved to: {outputFile}")


def generateReport(
    outputFile,
    datasetName,
    Xtrain,
    ytrain,
    Xtest,
    ytest,
    XtrainSklearn,
    XtestSklearn,
    customModel,
    customParams,
    customPreds,
    customAcc,
    sklearnModel,
    sklearnParams,
    sklearnPreds,
    sklearnAcc
):
    """
    Tries mlreport first.
    Falls back to matplotlib PDF if mlreport is unavailable or errors.
    """

    if MLREPORT_AVAILABLE:
        try:
            generateMLReport(
                outputFile=outputFile,
                datasetName=datasetName,
                Xtrain=Xtrain,
                ytrain=ytrain,
                Xtest=Xtest,
                ytest=ytest,
                XtrainSklearn=XtrainSklearn,
                XtestSklearn=XtestSklearn,
                customModel=customModel,
                customParams=customParams,
                customPreds=customPreds,
                sklearnModel=sklearnModel,
                sklearnParams=sklearnParams,
                sklearnPreds=sklearnPreds
            )
            return

        except Exception as e:
            print()
            print(f"[mlreport error: {e}]")
            print("Falling back to matplotlib PDF...")
            print()

    else:
        print()
        print("mlreport is not installed or could not be imported.")
        print("Falling back to matplotlib PDF...")
        print()

    fallbackMatplotlibReport(
        outputFile=outputFile,
        datasetName=datasetName,
        ytest=ytest,
        customParams=customParams,
        customAcc=customAcc,
        customPreds=customPreds,
        sklearnParams=sklearnParams,
        sklearnAcc=sklearnAcc,
        sklearnPreds=sklearnPreds
    )


def main():
    if len(sys.argv) < 2:
        print("Usage: python rfEval.py <CSVFile> [outputFile.pdf]")
        sys.exit(1)

    csvFile = sys.argv[1]

    if len(sys.argv) >= 3:
        outputFile = sys.argv[2]
    else:
        outputFile = "Report.pdf"

    print("Reading CSV file...")

    X, y, featureType, classVar = parseCSV(csvFile)

    if y is None:
        print("Error: rfEval.py requires a CSV file with class labels.")
        sys.exit(1)

    print(f"Dataset: {csvFile}")
    print(f"Number of rows: {len(X)}")
    print(f"Number of attributes: {len(X.columns)}")
    print(f"Class variable: {classVar}")

    print()
    print("Creating 80/20 train-test split...")

    Xtrain, ytrain, Xtest, ytest = trainTestSplit(
        X,
        y,
        testFrac=0.2,
        seed=67
    )

    print(f"Training rows: {len(Xtrain)}")
    print(f"Testing rows: {len(Xtest)}")

    grid = buildGrid(
        numAttrs=len(X.columns),
        numRows=len(Xtrain)
    )

    print()
    print("Grid settings:")
    print(grid)

    customModel, customParams, customAcc, customPreds = gridSearchCustomRF(
        Xtrain=Xtrain,
        ytrain=ytrain,
        Xtest=Xtest,
        ytest=ytest,
        featureType=featureType,
        grid=grid
    )

    print()
    print("===== Best Custom Random Forest =====")
    print("Best parameters:", customParams)
    print(f"Accuracy: {customAcc:.4f}")
    print(f"Error rate: {1 - customAcc:.4f}")

    print()
    print("Custom Random Forest Confusion Matrix:")
    printConfusionMatrix(ytest, customPreds)

    print()
    print("Encoding data for Scikit-learn...")

    XtrainSklearn, XtestSklearn = encodeForSklearn(
        Xtrain,
        Xtest,
        featureType
    )

    sklearnModel, sklearnParams, sklearnAcc, sklearnPreds = gridSearchSklearnRF(
        Xtrain=XtrainSklearn,
        ytrain=ytrain,
        Xtest=XtestSklearn,
        ytest=ytest,
        grid=grid
    )

    print()
    print("===== Best Scikit-learn Random Forest =====")
    print("Best parameters:", sklearnParams)
    print(f"Accuracy: {sklearnAcc:.4f}")
    print(f"Error rate: {1 - sklearnAcc:.4f}")

    print()
    print("Scikit-learn Random Forest Confusion Matrix:")
    printConfusionMatrix(ytest, sklearnPreds)

    print()
    print(f"Creating PDF report: {outputFile}")

    generateReport(
        outputFile=outputFile,
        datasetName=csvFile,
        Xtrain=Xtrain,
        ytrain=ytrain,
        Xtest=Xtest,
        ytest=ytest,
        XtrainSklearn=XtrainSklearn,
        XtestSklearn=XtestSklearn,
        customModel=customModel,
        customParams=customParams,
        customPreds=customPreds,
        customAcc=customAcc,
        sklearnModel=sklearnModel,
        sklearnParams=sklearnParams,
        sklearnPreds=sklearnPreds,
        sklearnAcc=sklearnAcc
    )

    print("Done!")


if __name__ == "__main__":
    main()

Writing rfEval.py


In [ ]:
!python rfEval.py iris.data.csv iris_report.pdf

Reading CSV file...
Dataset: iris.data.csv
Number of rows: 150
Number of attributes: 4
Class variable: species

Creating 80/20 train-test split...
Training rows: 120
Testing rows: 30

Grid settings:
{'NumTrees': [10, 25, 50, 100], 'NumAttributes': [1, 2, 3, 4], 'NumDataPoints': [0.3, 0.5, 0.7, 1.0]}

===== Custom Random Forest Grid Search =====
[1/64] N=10, m=1, k=0.3  acc=0.8667
[2/64] N=10, m=1, k=0.5  acc=0.7333
[3/64] N=10, m=1, k=0.7  acc=0.8333
[4/64] N=10, m=1, k=1.0  acc=0.8333
[5/64] N=10, m=2, k=0.3  acc=0.8667
[6/64] N=10, m=2, k=0.5  acc=0.9333
[7/64] N=10, m=2, k=0.7  acc=0.9000
[8/64] N=10, m=2, k=1.0  acc=0.9000
[9/64] N=10, m=3, k=0.3  acc=0.9000
[10/64] N=10, m=3, k=0.5  acc=0.9333
[11/64] N=10, m=3, k=0.7  acc=0.9000
[12/64] N=10, m=3, k=1.0  acc=0.9000
[13/64] N=10, m=4, k=0.3  acc=0.8667
[14/64] N=10, m=4, k=0.5  acc=0.9000
[15/64] N=10, m=4, k=0.7  acc=0.9000
[16/64] N=10, m=4, k=1.0  acc=0.9000
[17/64] N=25, m=1, k=0.3  acc=0.8333
[18/64] N=25, m=1, k=0.5  acc=0.8

In [ ]:
!python rfEval.py letter-recognition.data.csv letter_report.pdf

Reading CSV file...
Dataset: letter-recognition.data.csv
Number of rows: 20000
Number of attributes: 16
Class variable: lettr

Creating 80/20 train-test split...
Training rows: 16000
Testing rows: 4000

Grid settings:
{'NumTrees': [60, 120, 250, 500], 'NumAttributes': [1, 2, 3, 4, 5], 'NumDataPoints': [0.05, 0.1, 0.2, 0.25]}

===== Custom Random Forest Grid Search =====
[1/80] N=60, m=1, k=0.05  acc=0.3205
[2/80] N=60, m=1, k=0.1  acc=0.2810
[3/80] N=60, m=1, k=0.2  acc=0.2792
[4/80] N=60, m=1, k=0.25  acc=0.3147
[5/80] N=60, m=2, k=0.05  acc=0.5767
[6/80] N=60, m=2, k=0.1  acc=0.5877
[7/80] N=60, m=2, k=0.2  acc=0.5810
[8/80] N=60, m=2, k=0.25  acc=0.6172
[9/80] N=60, m=3, k=0.05  acc=0.7575
[10/80] N=60, m=3, k=0.1  acc=0.7582
[11/80] N=60, m=3, k=0.2  acc=0.7532
[12/80] N=60, m=3, k=0.25  acc=0.7875
[13/80] N=60, m=4, k=0.05  acc=0.8157
[14/80] N=60, m=4, k=0.1  acc=0.8442
[15/80] N=60, m=4, k=0.2  acc=0.8760
[16/80] N=60, m=4, k=0.25  acc=0.8690
[17/80] N=60, m=5, k=0.05  acc=0.850

In [ ]:
!python rfEval.py heart.csv heart_report.pdf

Reading CSV file...
Dataset: heart.csv
Number of rows: 918
Number of attributes: 11
Class variable: "HeartDisease"

Creating 80/20 train-test split...
Training rows: 734
Testing rows: 184

Grid settings:
{'NumTrees': [25, 50, 100, 200], 'NumAttributes': [1, 2, 3, 4, 5], 'NumDataPoints': [0.3, 0.5, 0.7, 1.0]}

===== Custom Random Forest Grid Search =====
[1/80] N=25, m=1, k=0.3  acc=0.7772
[2/80] N=25, m=1, k=0.5  acc=0.7880
[3/80] N=25, m=1, k=0.7  acc=0.7500
[4/80] N=25, m=1, k=1.0  acc=0.8315
[5/80] N=25, m=2, k=0.3  acc=0.8152
[6/80] N=25, m=2, k=0.5  acc=0.7935
[7/80] N=25, m=2, k=0.7  acc=0.7772
[8/80] N=25, m=2, k=1.0  acc=0.8207
[9/80] N=25, m=3, k=0.3  acc=0.8370
[10/80] N=25, m=3, k=0.5  acc=0.7989
[11/80] N=25, m=3, k=0.7  acc=0.8261
[12/80] N=25, m=3, k=1.0  acc=0.8478
[13/80] N=25, m=4, k=0.3  acc=0.8315
[14/80] N=25, m=4, k=0.5  acc=0.8587
[15/80] N=25, m=4, k=0.7  acc=0.8478
[16/80] N=25, m=4, k=1.0  acc=0.8152
[17/80] N=25, m=5, k=0.3  acc=0.8533
[18/80] N=25, m=5, k=0.5

In [ ]:
def generateMLReport(
    outputFile,
    datasetName,
    Xtrain, ytrain, Xtest, ytest,
    customModel,  customParams,  customPreds,
    sklearnModel, sklearnParams, sklearnPreds
):
    """
    Generates an mlreport ComparisonReport PDF.

    mlreport API (from https://github.com/Lucas-Summers/mlreport):

        Report(model, title, model_type, model_params)
            - model      : your fitted model object
            - model_type : "classification" (required for custom models)
            - model_params: dict of hyperparams shown in the report

        report.add_split("train", X_train, y_train, y_pred_train)
        report.add_split("test",  X_test,  y_test,  y_pred_test)
            - first arg is "train" or "test"
            - last arg is the predictions (y_pred)

        report.build()
            - computes all metrics and plots; must be called before export

        ComparisonReport(reports=[...], title=..., split="test", theme="light")
            - reports[0] is treated as the baseline
            - split tells it which split to compare on

        comparison.build()
        comparison.to_pdf("file.pdf")
    """

    #  Report for our custom Random Forest
    customReport = Report(
        customModel,
        title="Custom RandomForest",
        model_type="classification",
        model_params=customParams
    )
    customReport.add_split("train", Xtrain, list(ytrain), None)
    customReport.add_split("test",  Xtest,  list(ytest),  customPreds)
    customReport.build()

    # Report for Scikit-learn Random Forest
    sklearnReport = Report(
        sklearnModel,
        title="Sklearn RandomForest",
        model_type="classification",
        model_params=sklearnParams
    )
    sklearnReport.add_split("train", Xtrain, list(ytrain), None)
    sklearnReport.add_split("test",  Xtest,  list(ytest),  sklearnPreds)
    sklearnReport.build()

    # ComparisonReport
    comparison = ComparisonReport(
        reports=[customReport, sklearnReport],
        title=f"Random Forest Comparison — {datasetName}",
        split="test",
        theme="light"
    )
    comparison.build()
    comparison.to_pdf(outputFile)
    print(f"mlreport PDF saved to: {outputFile}")


def fallbackMatplotlibReport(
    outputFile,
    datasetName,
    ytest,
    customParams,  customAcc,  customPreds,
    sklearnParams, sklearnAcc, sklearnPreds
):
    """
    Fallback PDF using matplotlib if mlreport is not installed.
    """
    labels = sorted(
        set(list(ytest) + list(customPreds) + list(sklearnPreds)),
        key=str
    )

    with PdfPages(outputFile) as pdf:
        # Summary page
        fig = plt.figure(figsize=(8.5, 11))
        plt.axis("off")
        summaryText = (
            f"Random Forest Comparison Report\n\n"
            f"Dataset: {datasetName}\n\n"
            f"Custom Random Forest\n"
            f"  Accuracy:   {customAcc:.4f}\n"
            f"  Error Rate: {1 - customAcc:.4f}\n"
            f"  Params: {customParams}\n\n"
            f"Scikit-learn Random Forest\n"
            f"  Accuracy:   {sklearnAcc:.4f}\n"
            f"  Error Rate: {1 - sklearnAcc:.4f}\n"
            f"  Params: {sklearnParams}\n"
        )
        plt.text(0.05, 0.95, summaryText, va="top", fontsize=10,
                 family="monospace")
        pdf.savefig(fig)
        plt.close(fig)

        # Custom RF confusion matrix
        cmCustom = confusion_matrix(ytest, customPreds, labels=labels)
        fig = plt.figure(figsize=(8, 6))
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cmCustom,
            display_labels=labels
        )
        disp.plot(xticks_rotation=45)
        plt.title("Custom Random Forest — Confusion Matrix")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

        # Sklearn confusion matrix
        cmSklearn = confusion_matrix(ytest, sklearnPreds, labels=labels)
        fig = plt.figure(figsize=(8, 6))
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cmSklearn,
            display_labels=labels
        )
        disp.plot(xticks_rotation=45)
        plt.title("Scikit-learn Random Forest — Confusion Matrix")
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)

    print(f"Fallback matplotlib PDF saved to: {outputFile}")


def generateReport(
    outputFile,
    datasetName,
    Xtrain, ytrain, Xtest, ytest,
    customModel,  customParams,  customPreds,  customAcc,
    sklearnModel, sklearnParams, sklearnPreds, sklearnAcc
):
    """
    Tries mlreport first. Falls back to matplotlib if mlreport is missing.
    """
    if MLREPORT_AVAILABLE:
        try:
            generateMLReport(
                outputFile, datasetName,
                Xtrain, ytrain, Xtest, ytest,
                customModel,  customParams,  customPreds,
                sklearnModel, sklearnParams, sklearnPreds
            )
            return
        except Exception as e:
            print(f"[mlreport error: {e}]")
            print("Falling back to matplotlib PDF...\n")

    fallbackMatplotlibReport(
        outputFile, datasetName, ytest,
        customParams,  customAcc,  customPreds,
        sklearnParams, sklearnAcc, sklearnPreds
    )

In [ ]:
%%writefile README.md

# CSC 466 Lab 3 - Random Forest Classifier

# Team Members
Name: Winnie Trinh
Email: witrinh@calpoly.edu

Name: Nathan Madlansacay
Email: nmadlans@calpoly.edu

# Project Overview

This project extends our Lab 2 C4.5 decision tree implementation into a Random Forest classifier. The custom Random Forest builds multiple C4.5 decision trees using bootstrap samples of the training data and random subsets of attributes. Predictions are made by collecting votes from all trees and choosing the majority class. The main Lab 3 evaluation program is `rfEval.py`. It performs an 80/20 train-test split, runs a grid search over Random Forest hyperparameters, compares our custom Random Forest against Scikit-learn's `RandomForestClassifier`, and generates a PDF report. This submission includes both our Lab 2 files and Lab 3 files because Lab 3 builds on the C4.5 implementation from Lab 2.

# Submitted Files

# Lab 2 Files

# c45-lab02.py
Contains our implementation of the C4.5 decision tree classifier. This file includes:
- The c45 class
- Entropy calculation
- Information Gain calculation
- Gain Ratio calculation
- Numeric and categorical splitting
- Recursive decision tree construction using .fit()
- Prediction using .predict()
- Saving decision trees as JSON with .save_tree()
- Loading decision trees from JSON with .read_tree()

# InduceC45-lab02.py
Command-line program for training a C4.5 decision tree on a full dataset.
This program:
- Reads a CSV file in the required Lab 2/Lab 3 format
- Parses the feature columns and class variable
- Trains a C4.5 decision tree using InfoGain and threshold 0.0
- Prints the trained tree in JSON format
- Optionally saves the trained tree to a JSON file

Usage:
```bash
python InduceC45.py <TrainingSetFile.csv> [outputTree.json]
```

# predict-lab02.py
Command-line program for using a saved C4.5 decision tree to make predictions.
This program:
- Loads a saved decision tree from a JSON file
- Reads a CSV file
- Prints one prediction per row
- Optionally runs in evaluation mode using the eval argument
- In evaluation mode, prints actual labels, predicted labels, accuracy, error rate, and a confusion matrix

Usage:
```bash
python predict.py <CSVFile> <JSONFile> [eval]
```

# crossVal-lab02.py
Runs 10-fold cross-validation for our custom C4.5 implementation.
This program:
- Reads a dataset CSV file
- Reads a JSON grid file containing threshold values
- Tests both InfoGain and Ratio
- Runs 10-fold cross-validation for each hyperparameter setting
- Prints the best splitting metric, best threshold, overall cross-validation accuracy, and confusion matrix

Usage:
```bash
python crossVal.py <CSVFile> <GridFile>
```

# crossValSKL-lab02.py
Runs 10-fold cross-validation using Scikit-learn's DecisionTreeClassifier.
This program:
- Reads the same CSV format as the custom implementation
- Converts categorical columns using Scikit-learn preprocessing
- Uses Scikit-learn's entropy criterion to match Information Gain
- Runs 10-fold cross-validation
- Prints the best threshold, accuracy, and confusion matrix
- Optionally saves the final Scikit-learn decision tree visualization as an image

Usage:
```bash
python crossValSKL.py <CSVFile> <GridFile> [outputTree.png]
```

# grid-lab02.json
Contains the hyperparameter grid used by the Lab 2 cross-validation programs.
The file contains threshold values for:
- InfoGain
- Ratio

Current grid:
```json
{
  "InfoGain": [0.0, 0.01, 0.05, 0.1],
  "Ratio": [0.0, 0.01, 0.05, 0.1]
}
```

# utils-lab02.py
Contains helper functions used by both Lab 2 and Lab 3 programs.
This file includes:
- parseCSV() for reading the custom CSV format
- Train-test split helpers
- Accuracy and error rate helpers
- Dynamic grid creation for Random Forest hyperparameter tuning
- Encoding helpers for Scikit-learn
- Confusion matrix helper functions

# Lab 3 Files

# RandomForestClassifier.py
Contains our custom Random Forest classifier.
This file includes:
- The RandomForestClassifier class
- Bootstrap sampling of rows with replacement
- Random attribute selection without replacement
- Training multiple C4.5 decision trees
- Majority-vote prediction across all trees
- Consistent tie-breaking using string order

The Random Forest constructor supports the following hyperparameters:
- NumTrees: number of decision trees in the forest
- NumAttributes: number of randomly selected attributes per tree
- NumDataPoints: number or fraction of rows sampled for each tree
- SplittingMetric: C4.5 splitting metric, such as InfoGain
- Threshold: C4.5 splitting threshold
- random_state: random seed for reproducibility

#### rfEval.py
Main evaluation program for Lab 3.
This program:
- Reads a dataset CSV file
- Creates an 80/20 train-test split
- Builds a dynamic grid for Random Forest hyperparameter search
- Runs grid search on our custom Random Forest
- Runs grid search on Scikit-learn's Random Forest
- Prints the best parameters, accuracy, error rate, and confusion matrix for both models
- Generates a PDF comparison report

The program first attempts to generate a report using mlreport. If mlreport is unavailable or causes an error, the program creates a fallback PDF report using Matplotlib.

Usage:
```bash
python rfEval.py <CSVFile> [outputFile.pdf]
```

If no output PDF filename is provided, the program creates:
```bash
Report.pdf
```

# Input File Format
All dataset CSV files must use the modified three-line header format used in the lab.

# Line 1: Column Names
The first line contains the names of all columns.

# Line 2: Domain Codes
The second line contains a comma-separated list of domain codes.
The domain codes mean:
- -1: row ID or metadata column; ignored by the classifier
- 0: numeric variable
- Positive integer: categorical variable

# Line 3: Class Variable
The third line contains the name of the class variable.

# Remaining Lines: Data
All remaining lines contain the actual comma-separated data rows.

# Random Forest Grid Search
For Lab 3, no separate grid search settings file is required. The grid search ranges are created dynamically inside rfEval.py based on the size of the dataset.
The Random Forest grid search tunes:
- NumTrees
- NumAttributes
- NumDataPoints

The C4.5 pass-through hyperparameters are fixed as:
```text
SplittingMetric = InfoGain
Threshold = 0.0
```

The same grid is used to compare:
1. Our custom Random Forest implementation
2. Scikit-learn's Random Forest implementation

# Output
When rfEval.py runs, it prints:
- Dataset name
- Number of rows
- Number of attributes
- Class variable
- Train/test split sizes
- Grid settings
- Grid search progress
- Best custom Random Forest parameters
- Custom Random Forest accuracy and error rate
- Custom Random Forest confusion matrix
- Best Scikit-learn Random Forest parameters
- Scikit-learn accuracy and error rate
- Scikit-learn confusion matrix

It also creates a PDF comparison report.

## Required Packages
The programs use the following Python packages:
```bash
pandas
scikit-learn
matplotlib
mlreport
```

## Notes for the Grader

- Our Lab 3 Random Forest implementation uses our Lab 2 C4.5 implementation to build each decision tree.
- The Random Forest uses bootstrap sampling for rows and random sampling without replacement for attributes.
- The validation method for Lab 3 is an 80/20 train-test split.
- The random seed is fixed for reproducibility.
- Categorical features are handled directly by our custom C4.5 implementation.
- For Scikit-learn models, categorical features are converted into numeric/dummy variables before training.
- The PDF reports for Iris, Letter Recognition, and Heart Disease are submitted separately to Gradescope.
- The handin submission should include this `README.md` file directly and the code archive named `lab03.zip`.

Overwriting README.md
